## Using edgeR for DEG analysis between regions -- comparing region_leiden clusters within capillary cells and using pseurod-bulk approach

In [ ]:
### identify cluster marker genes
suppressPackageStartupMessages({
    library(Seurat)
    library(stringr)
    library(dplyr)
    library(patchwork)
    library(ggplot2)
    library(future)
    library(tidydr)
    library(harmony)
    library(reticulate)
    library(viridis)
    library(RColorBrewer)
    library(ComplexHeatmap)
    library(colorRamp2)
    library(edgeR)
})

# set large memory  
options(future.globals.maxSize = 120*1024^3)

## plotting parameters
umap_theme <- theme_dr()+theme(panel.grid.major = element_blank(), 
                                            panel.grid.minor = element_blank(),
                                            panel.background = element_blank(), 
                                            axis.line = element_line(colour = "black"))

## set working dir
dir = "/home/kibr/Work/Vascular_atlas"
setwd(dir)

In [ ]:
## Reloading 
# Convert h5ad to h5seurat
h5ad_file="./Results/Revision_2/Capillary_temp_processed.h5ad"
obj_oi = schard::h5ad2seurat(h5ad_file)
print(obj_oi)

## 
### ---------------------------------Drawing heatmap of transporter gene expression ----------------------------------------

In [ ]:
### Making sure get the corrected layer and matrix
DefaultAssay(obj_oi) = "RNA"
# obj_oi = JoinLayers(obj_oi)

### Aggregation to pseudobulk 
mtx = AggregateExpression(
        obj_oi, 
        # features = gene_oi,
        group.by = c("region_name","individualID"),
        assays = 'RNA',
        slot = "counts"
    ) 
mtx <- as.matrix(mtx$RNA)

## Get the library size for each sample
lib_size <- Matrix::colSums(mtx)

### Get the logCPM from the matrix
cpm <- t(t(mtx) / lib_size) * 1e6
logCPM <- log2(cpm + 1)

## Only focus on the genes of interest
gene_oi = c("ABCC1","ABCB1","ABCG2","SLC16A1","SLC2A1","SLC7A5","TFRC","INSR","IGF1R","CLDN5","OCLN","TJP1", "CAV1",'SLC7A1',"SLC22A5")

logCPM = logCPM[gene_oi,]
logCPM_z <- t(scale(t(logCPM)))

### Calculate the averaged logCPM
# extract region (everything before first "_")
region <- sub("_.*$", "", colnames(logCPM))

# sanity check
table(region)

logCPM_region <- sapply(
  split(seq_len(ncol(logCPM)), region),
  function(i) {
    rowMeans(logCPM[, i, drop = FALSE])
  }
)
logCPM_region_z <- t(scale(t(logCPM_region)))

## Calculate the neuronal density of each region/sample

In [ ]:
h5ad_file="./Results/Revision_2/clean_object_revision_03032026.h5ad"
clean = schard::h5ad2seurat(h5ad_file)
print(clean)

In [ ]:
length(unique(clean$region_name))

In [ ]:
meta = clean@meta.data

cell_counts = meta %>% 
                    filter(Cell_class == "Neuron") %>%
                    group_by(region_name,individualID) %>%
                    summarise(cell_counts = n())

# Also count total cells per region-sample
total_counts <- meta %>%
  group_by(region_name, individualID) %>%
  summarise(total_count = n())

cell_density = left_join(cell_counts, total_counts, by = c("region_name","individualID")) %>%
                    mutate(cell_density = cell_counts/total_count)
head(cell_density)

In [ ]:
avg_density <- cell_density %>%
  group_by(region_name) %>%
  summarise(mean_cell_density = mean(cell_density, na.rm = TRUE)) %>%
  mutate(region_name = trimws(stringr::str_to_title(region_name))) %>%
  arrange(factor(region_name, levels = colnames(logCPM_region_z)))
avg_density[avg_density$region_name == "Mid Temporal Gyrus",]$region_name = "Middle Temporal Gyrus"
avg_density = avg_density[avg_density$region_name %in% colnames(logCPM_region_z),]

# Ensure avg_density is a named vector matching the column names of mtx
avg_vec <- avg_density$mean_cell_density
names(avg_vec) <- avg_density$region_name
avg_vec <- avg_vec[colnames(logCPM_region_z)]  # match order to matrix columns

In [ ]:
## Provide the color code
meta <- obj_oi@meta.data
df <- unique(meta[,c("region_name","region_layer")])

df = as.data.frame(df)

df$region_color <- NA
df[df$region_layer == "Cerebral Cortex",]$region_color = "#009E73"
df[df$region_layer == "Brain Stem and Spinal Cord",]$region_color = "#D55E00"
df[df$region_layer == "Cerebellum",]$region_color = "#0072B2"
df[df$region_layer == "Subcortical Region",]$region_color = "#56B4E9"
df[df$region_layer == "Cerebral Cortex Watershed",]$region_color = "#E69F00"
df[df$region_layer == "White Matter",]$region_color = "#CC79A7"
df[df$region_layer == "Olfactory Bulb",]$region_color = "#999999"

# ## Assign region color
region_color <- df$region_color
names(region_color) <-df$region_name

# annotation for heatmap
hl <- HeatmapAnnotation(
  region = colnames(logCPM_region_z),
  avg_density = avg_vec,
  col = list(
    region = region_color,
    avg_density = colorRamp2(
      c(min(avg_vec, na.rm = TRUE), max(avg_vec, na.rm = TRUE)),
      c("white", "#08415C")
    )
  ),
  annotation_name_side = "right"
)

# Bottom annotation: rotated brain region labels
bn <- HeatmapAnnotation(
  text = anno_text(
    colnames(logCPM_region_z),
    rot = 90,
    offset = unit(1, "npc"),
    just = "right"
  ),
  annotation_height = max_text_width(colnames(logCPM_region_z))
)
# Main heatmap
ht <- Heatmap(
  logCPM_region_z,
  cluster_rows = TRUE,
  cluster_columns = TRUE,
  column_km = 5,
#   row_km = 4,
  col = colorRamp2(c(-2.5,0,2.5), c("#5387be","#F7F7F7","#b31f2c")),
  top_annotation = hl,
  bottom_annotation = bn,
  show_column_names = FALSE,
  show_row_names = TRUE,
  show_heatmap_legend = TRUE,
  use_raster = TRUE#,
  # cell_fun = function(j, i, x, y, w, h, fill) {
  #   if (sig_mat[i, j] < 0.05) {
  #     grid.text("*", x, y, gp = gpar(fontsize = 20, col = "black"), vjust = "center")
  #   }
  # }
)

In [ ]:
# Set display options (works in Jupyter or R notebooks)
options(repr.plot.width = 14, repr.plot.height = 7, repr.plot.res = 200)
set.seed(42)

# pdf(file = "./Results/Revision_2/Figures/Figure_5X_Capillary_BBB_signature_heatmap_per_region.pdf", width = 13, height = 7)
# ht
# dev.off()
ht

## Calculate the scaled and averaged expression of selected genes

In [ ]:
clean$region_name= trimws(stringr::str_to_title(clean$region_name))
clean$region_name[clean$region_name == "Mid Temporal Gyrus"] = "Middle Temporal Gyrus"
table(clean$region_name)

In [ ]:
### For capillary first
## Only focus on the genes of interest
obj = subset(clean,subset = Cell_class == "Endothelial")
genes_oi = c("ABCC1","ABCB1","ABCG2","SLC16A1","SLC2A1","SLC7A5","TFRC","INSR","IGF1R","CLDN5","OCLN","TJP1", "CAV1",'SLC7A1',"SLC22A5")

### Making sure get the corrected layer and matrix
DefaultAssay(obj) = "RNA"
obj = NormalizeData(obj)
obj <- ScaleData(obj,features = genes_oi)

mt <- AverageExpression(obj, features = genes_oi,group.by = c("region_name"),layer = "scale.data",assays = "RNA")
# # mt = AggregateExpression(
# #         obj_oi, 
# #         # features = genes_oi,
# #         group.by = c("region_name","individualID"),
# #         assays = 'RNA',
# #         slot = "counts"
# #     )
mt <- as.matrix(mt$RNA)
# ### remove the g from the colnames
# colnames(mt) <- substring(colnames(mt),2)
# colnames(mt) <- gsub("-","_",colnames(mt))

### get the neuronal density

In [ ]:
## Get the meta data for the clean object to calculate neuron density
meta = clean@meta.data
ct_oi = "Neuron"
cell_counts = meta %>% 
                    filter(Cell_class == ct_oi) %>%
                    group_by(region_name,individualID) %>%
                    summarise(cell_counts = n())

# Also count total cells per region-sample
total_counts <- meta %>%
  group_by(region_name , individualID) %>%
  summarise(total_count = n())

cell_density = left_join(cell_counts, total_counts, by = c("region_name","individualID")) %>%
                    mutate(cell_density = cell_counts/total_count)

avg_density <- cell_density %>%
  group_by(region_name) %>%
  summarise(mean_cell_density = mean(cell_density, na.rm = TRUE)) %>%
  arrange(factor(region_name, levels = colnames(mtx)))

avg_density$region_name = gsub("-","_",avg_density$region_name)
avg_density = avg_density[avg_density$region_name %in% colnames(mt),]

# Ensure avg_density is a named vector matching the column names of mtx
avg_vec <- avg_density$mean_cell_density
names(avg_vec) <- avg_density$region_name
avg_vec <- avg_vec[colnames(mt)]  # match order to matrix columns

In [ ]:
## Matrix with neuron density and logCPM which were calcuated above
mtx = rbind(mt, avg_neuronal_density = avg_vec)
mtx

### get the astrocyte density

In [ ]:
## Get the meta data for the clean object to calculate neuron density
meta = clean@meta.data
ct_oi = "Astrocyte"
cell_counts = meta %>% 
                    filter(Cell_class == ct_oi) %>%
                    group_by(region_name,individualID) %>%
                    summarise(cell_counts = n())

# Also count total cells per region-sample
total_counts <- meta %>%
  group_by(region_name , individualID) %>%
  summarise(total_count = n())

cell_density = left_join(cell_counts, total_counts, by = c("region_name","individualID")) %>%
                    mutate(cell_density = cell_counts/total_count)

avg_density <- cell_density %>%
  group_by(region_name) %>%
  summarise(mean_cell_density = mean(cell_density, na.rm = TRUE)) %>%
  arrange(factor(region_name, levels = colnames(mtx)))

avg_density$region_name = gsub("-","_",avg_density$region_name)
avg_density = avg_density[avg_density$region_name %in% colnames(mt),]

# Ensure avg_density is a named vector matching the column names of mtx
avg_vec <- avg_density$mean_cell_density
names(avg_vec) <- avg_density$region_name
avg_vec <- avg_vec[colnames(mt)]  # match order to matrix columns

In [ ]:
## Matrix with astrocyte density and logCPM
mtx = rbind(mtx, avg_astrocyte_density = avg_vec)
tail(mtx)

### Get the genes_oi for neurons

In [ ]:
### For capillary first
## Only focus on the genes of interest
obj = subset(clean,subset = Cell_class == "Neuron")
genes_oi = c("SYT1","ROBO2","NRXN1","GAD2","FOS","RBFOX3")

### Making sure get the corrected layer and matrix
DefaultAssay(obj) = "RNA"
obj = NormalizeData(obj)
obj <- ScaleData(obj,features = genes_oi)

mt <- AverageExpression(obj, features = genes_oi,group.by = c("region_name"),layer = "scale.data",assays = "RNA")
# # mt = AggregateExpression(
# #         obj_oi, 
# #         # features = genes_oi,
# #         group.by = c("region_name","individualID"),
# #         assays = 'RNA',
# #         slot = "counts"
# #     )
mt <- as.matrix(mt$RNA)
### remove the g from the colnames
# colnames(mt) <- substring(colnames(mt),2)
# colnames(mt) <- gsub("-","_",colnames(mt))

In [ ]:
combined = merge.data.frame(t(mtx), t(mt), by = "row.names", all = TRUE)
rownames(combined) = combined$Row.names
combined$Row.names = NULL
combined

### calculate the genes of interest in astrocytes

In [ ]:
### For capillary first
## Only focus on the genes of interest
obj = subset(clean,subset = Cell_class == "Astrocyte")
genes_oi = c("SLC1A2","GFAP","RFX4","NRG3","AQP4","NDRG2","ALDH1L1")

### Making sure get the corrected layer and matrix
DefaultAssay(obj) = "RNA"
obj = NormalizeData(obj)
obj <- ScaleData(obj,features = genes_oi)

mt <- AverageExpression(obj, features = genes_oi,group.by = c("region_name"),layer = "scale.data",assays = "RNA")
# # mt = AggregateExpression(
# #         obj_oi, 
# #         # features = genes_oi,
# #         group.by = c("region_name","individualID"),
# #         assays = 'RNA',
# #         slot = "counts"
# #     )
mt <- as.matrix(mt$RNA)
### remove the g from the colnames
# colnames(mt) <- substring(colnames(mt),2)
# colnames(mt) <- gsub("-","_",colnames(mt))

In [ ]:
combined = merge.data.frame(combined, t(mt), by = "row.names", all = TRUE)
rownames(combined) = combined$Row.names
combined$Row.names = NULL
combined

In [ ]:
## fill NA with 0
combined[is.na(combined)] <- 0
combined

In [ ]:
write.csv(combined, file = "./Results/Revision_2/combined_expression_density.csv", row.names = TRUE)

In [ ]:
combined = read.csv("./Results/Revision_2/Capillary_BBB_signature_and_cell_density_per_region.csv", row.names = 1)
## scale each column
combined_scaled <- as.data.frame(scale(combined))
rownames(combined_scaled) = rownames(combined)
combined_scaled

In [ ]:
# region_meta = read.csv("./Data/2_Vascular_Mapped_Region_ID.csv")
# region_meta$region_name = trimws(stringr::str_to_title(region_meta$Region_layer_1))
# region_meta
# write.csv(region_meta, "./Data/2_Vascular_Mapped_Region_ID.csv", row.names = FALSE)

In [ ]:
# ## save the combined matrix
# write.csv(combined_scaled, "./Results/Revision_2/Capillary_BBB_signature_and_cell_density_per_region_scaled.csv", row.names = TRUE)

## Calculate correlation

In [ ]:
library(DESeq2)
# Function to create pseudobulk matrix for a single cell type
CreatePseudoBulk <- function(seurat_obj, target_celltype, assay){
    seurat_sub <- subset(seurat_obj, subset = !!sym(celltype_col) == target_celltype)
    exp_mat <- AggregateExpression(object = seurat_sub,assays = assay,group.by = c("individualID","region_name"))$RNA

    # For raw counts
    keep_genes <- rowSums(exp_mat > 0) >= 5
    exp_mat <- exp_mat[keep_genes, ]
    return(exp_mat)
}

## preparing vsd matrix
CreateVSTMatrix <- function(mat= mat,var.per=0.95){
    sample_metadata = as.data.frame(colnames(mat))
    sample_metadata$individualID = str_split_fixed(sample_metadata[,1],pattern = "_",n = 2)[,1]
    sample_metadata$brain_region = str_split_fixed(sample_metadata[,1],pattern = "_",n = 2)[,2]

    dds = DESeqDataSetFromMatrix(countData = mat,
                                  colData = sample_metadata,
                                  design = ~1)
    dds = estimateSizeFactors(dds,type = "poscounts")           
    vsd = vst(dds, blind = TRUE)
    # vsd = rlog(dds, blind = TRUE)
    vst_matrix = assay(vsd)

    ## only keeping the highly variable genes
    row_var <- rowVars(vst_matrix)
    q95_var <- quantile(row_var, var.per)
    vst_matrix <- vst_matrix[row_var> q95_var, ]

    return(vst_matrix)
}

In [ ]:
## merge the layers and normalizing the data
DefaultAssay(clean) = "RNA"
clean <- NormalizeData(clean)

print(clean)

In [ ]:
# Choose the assay and slot
assay_use = "RNA"
celltype_col = "Cell_class"

ct_1 = "Endothelial"
ct_2 = "Neuron"
# ct_2 = "Astrocyte"

mat_1 = CreatePseudoBulk(seurat_obj = clean, target_celltype = ct_1, assay = "RNA")
mat_2 = CreatePseudoBulk(seurat_obj = clean, target_celltype = ct_2, assay = "RNA")

In [ ]:
# Align on common samples
common_samples <- intersect(colnames(mat_1), colnames(mat_2))
mat_1 <- mat_1[, common_samples]
mat_2 <- mat_2[, common_samples]

print(dim(mat_1))
print(dim(mat_2))

vst_mat_1 = CreateVSTMatrix(mat = mat_1,var.per = 0.5)
vst_mat_2 = CreateVSTMatrix(mat = mat_2,var.per = 0.5)

In [ ]:
vst_mat_1 = vst_mat_1[c("SLC2A1","INSR","TFRC","SLC16A1"),]
# vst_mat_2 = vst_mat_2[c("SLC1A2","GFAP","RFX4","NRG3","ALDH1L1","NDRG2"),]
vst_mat_2 = vst_mat_2[c("SYT1","ROBO2","NRXN1","GAD2","FOS","RBFOX3"),]

In [ ]:
library(Hmisc)
rownames(vst_mat_1) <- paste0(rownames(vst_mat_1), "_A")
rownames(vst_mat_2) <- paste0(rownames(vst_mat_2), "_B")

## Running the spearman correlation analysis on the two gene lists from two cell types. 
combined <- cbind(t(vst_mat_1), t(vst_mat_2))  # samples × genes

# rcorr() computes correlations and p-values
result <- rcorr(as.matrix(combined), type = "spearman")  # or "pearson"

# Extract relevant correlation block (astro × capillary only)
n_1 <- nrow(vst_mat_1)
n_2 <- nrow(vst_mat_2)

cor_mat <- result$r[1:n_1, (n_1 + 1):(n_1 + n_2)]
p_mat   <- result$P[1:n_1, (n_1 + 1):(n_1 + n_2)]

In [ ]:
# Flatten to vector
pvals <- as.vector(p_mat)

# Adjust p-values (FDR)
padj <- p.adjust(pvals, method = "bonferroni")

# Reshape to original matrix dimensions
padj_mat <- matrix(padj, nrow = nrow(p_mat), ncol = ncol(p_mat),
                   dimnames = dimnames(p_mat))

sig_pairs <- which(padj_mat < 0.05, arr.ind = TRUE)

df_long <- data.frame(
  gene_A = rep(rownames(cor_mat), times = ncol(cor_mat)),
  gene_B = rep(colnames(cor_mat), each  = nrow(cor_mat)),
  correlation = as.vector(cor_mat),
  p_value     = as.vector(p_mat),
  padj        = as.vector(padj_mat)
)


df_sig <- subset(df_long, padj < 0.05)
df_sig[order(df_sig$correlation,decreasing = T),]

In [ ]:
# Replace these with your actual gene names
gene1 <- "SLC16A1"
gene2 <- "SYT1"

rownames(vst_mat_1) = gsub("_A","",rownames(vst_mat_1))
rownames(vst_mat_2) = gsub("_B","",rownames(vst_mat_2))

# Extract expression vectors
x <- as.numeric(vst_mat_1[gene1, ])
y <- as.numeric(vst_mat_2[gene2, ])

# Optional: name the points
labels <- colnames(vst_mat_2)

df <- data.frame(
  gene1 = x,
  gene2 = y,
  region = labels
)


# Fit a linear model
model <- glm(gene2 ~ gene1, data = df, family = gaussian())
# Summary of model
summary(model)

options(repr.plot.height = 6,repr.plot.width = 7)
p = ggplot(df, aes(x = gene1, y = gene2, label = region)) +
  geom_point(color = "blue", size = 3) +
  geom_smooth(method = "lm", se = TRUE, color = "darkred") +
  # geom_text(nudge_y = 0.05, size = 3, alpha = 0.6) +  # optional labels
  labs(
    x = paste(gene1, "Expression in",ct_1),
    y = paste(gene2, "Expression in",ct_2),
    title = paste("Correlation of", gene1, "vs", gene2)
  ) +
  theme_minimal() +
  theme(
    axis.title = element_text(size = 16),
    axis.text = element_text(size = 14),
    plot.title = element_text(size = 18, face = "bold"),
    panel.border = element_rect(color = "black", fill = NA, linewidth = 1)  # <- bezel-style outlin
  )
## Add correlation coefficient and p-value to the plot
corr = cor.test(x, y, method = "spearman")
p_value <- signif(corr$p.value, 3)
cor_coef <- round(corr$estimate, 3)
## save plot
p = p + annotate("text", x = max(x), y = max(y), label = paste("r2 =", cor_coef, "\np =", p_value), hjust = 1, vjust = 12)

## saving
ggsave(filename = "./Results/Revision_2/Figures/Figure_5X_Capillary_BBB_signature_correlation_with_neuronal_genes.pdf", plot = p, width = 7, height = 6, device = "pdf")
# p

In [ ]:
sessionInfo()